# Day 5: Data Manipulation and Analysis
## Notebook 3: Grouping, Aggregation, and Reshaping

**Introduction to Scientific Programming**  
CNC-UC

This notebook covers:
- Basic GroupBy operations
- Advanced aggregation
- Pivot tables
- Long vs wide format conversion
- Concatenating and merging DataFrames

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

print("Setup complete!")

## 1. Basic GroupBy Operations

In [ ]:
# Create example experimental data
np.random.seed(42)
n_subjects = 3
n_conditions = 3
n_trials = 10

subjects = []
conditions = []
rts = []
accuracies = []

for subj in range(1, n_subjects + 1):
    for cond in ['control', 'drug_A', 'drug_B']:
        for trial in range(n_trials):
            subjects.append(f'S{subj:02d}')
            conditions.append(cond)
            
            # Drug effects
            base_rt = 500
            if cond == 'drug_A':
                base_rt = 450
            elif cond == 'drug_B':
                base_rt = 420
            
            rts.append(np.random.gamma(2, base_rt/2))
            accuracies.append(np.random.binomial(1, 0.85))

df = pd.DataFrame({
    'subject': subjects,
    'condition': conditions,
    'reaction_time': rts,
    'accuracy': accuracies
})

print(f"Created dataset with {len(df)} trials")
df.head(10)

### Single grouping variable

In [ ]:
# Group by condition
grouped = df.groupby('condition')

# Common aggregations
print("Mean reaction time by condition:")
df.groupby('condition')['reaction_time'].mean()

In [ ]:
print("Count per condition:")
df.groupby('condition')['reaction_time'].count()

In [ ]:

print("Summary statistics by condition:")
df.groupby('condition')['reaction_time'].describe()

### Multiple grouping variables

In [ ]:
# Group by subject and condition
summary = df.groupby(['subject', 'condition']).agg({
    'reaction_time': 'mean',
    'accuracy': 'mean'
})

print("Mean RT and accuracy per subject and condition:")
summary

### Iterate through groups

In [ ]:
# Access individual groups
for name, group in df.groupby('condition'):
    print(f"Condition: {name}")
    print(f"  N trials: {len(group)}")
    print(f"  Mean RT: {group['reaction_time'].mean():.2f}")
    print(f"  Mean accuracy: {group['accuracy'].mean():.2f}")
    print()

### Multiple aggregations with .agg()

In [ ]:
# Multiple aggregations
result = df.groupby('condition')['reaction_time'].agg(['mean', 'std', 'count'])
print("Multiple aggregations:")
result

### Different aggregations per column

In [ ]:
# Different aggregations for different columns
result = df.groupby('condition').agg({
    'reaction_time': ['mean', 'std'],
    'accuracy': ['mean', 'sum']
})

print("Different aggregations per column:")
result

### Named aggregations (clearer column names)

In [ ]:
# Named aggregations
result = df.groupby('condition').agg(
    mean_rt=('reaction_time', 'mean'),
    std_rt=('reaction_time', 'std'),
    n_trials=('reaction_time', 'count'),
    total_correct=('accuracy', 'sum')
).reset_index()

print("Named aggregations:")
result

### Custom aggregation functions

In [ ]:
# Custom functions
def range_func(x):
    return x.max() - x.min()

def coefficient_of_variation(x):
    return x.std() / x.mean()

result = df.groupby('condition').agg({
    'reaction_time': ['mean', range_func, coefficient_of_variation]
})

print("Custom aggregation functions:")
result

### Transform: keep original DataFrame shape

In [ ]:
# Transform broadcasts group statistics back to all rows
df['condition_mean'] = df.groupby('condition')['reaction_time'].transform('mean')
df['condition_std'] = df.groupby('condition')['reaction_time'].transform('std')

# Calculate z-score within condition
df['z_score_within_condition'] = (
    (df['reaction_time'] - df['condition_mean']) / df['condition_std']
)

print("Transform example (z-score within condition):")
df[['subject', 'condition', 'reaction_time', 'condition_mean', 'z_score_within_condition']].head(15)

### Filter: keep/remove entire groups

In [ ]:
# Keep only groups with > 15 observations
df_filtered = df.groupby('subject').filter(lambda x: len(x) > 15)

print(f"Original: {len(df)} rows")
print(f"After filter: {len(df_filtered)} rows")
print("\nSubjects remaining:")
df_filtered['subject'].unique()

## 3. Pivot Tables

In [ ]:
# Basic pivot table
pivot = pd.pivot_table(
    df,
    values='reaction_time',
    index='subject',
    columns='condition',
    aggfunc='mean'
)

print("Pivot table (subject × condition):")
pivot

In [ ]:
# Multiple aggregations
pivot_multi = pd.pivot_table(
    df,
    values='reaction_time',
    index='subject',
    columns='condition',
    aggfunc=['mean', 'std', 'count']
)

print("Pivot table with multiple aggregations:")
pivot_multi

In [ ]:
# Add margins (row/column totals)
pivot_margins = pd.pivot_table(
    df,
    values='reaction_time',
    index='subject',
    columns='condition',
    aggfunc='mean',
    margins=True,
    margins_name='Overall'
)

print("Pivot table with margins:")
pivot_margins

In [ ]:
# Crosstab: frequency counts
df['response_type'] = np.where(df['accuracy'] == 1, 'correct', 'incorrect')

crosstab = pd.crosstab(df['condition'], df['response_type'])
print("Crosstab (condition × response):")
crosstab

## 4. Long vs Wide Format

### Wide to Long: pd.melt()

In [ ]:
# Create wide format data
df_wide = pd.DataFrame({
    'subject': ['S01', 'S02', 'S03'],
    'pre_test': [85, 78, 92],
    'post_test': [92, 88, 95]
})

print("Wide format:")
print(df_wide)
print()

# Wide to long
df_long = pd.melt(
    df_wide,
    id_vars=['subject'],
    value_vars=['pre_test', 'post_test'],
    var_name='timepoint',
    value_name='score'
)

print("Long format:")
df_long

### Long to Wide: pd.pivot()

In [ ]:
# Long to wide
df_back_to_wide = df_long.pivot(
    index='subject',
    columns='timepoint',
    values='score'
)

print("Back to wide format:")
df_back_to_wide

### Multiple value columns

In [ ]:
# Wide format with multiple measurements
df_multi_wide = pd.DataFrame({
    'subject': ['S01', 'S02', 'S03'],
    'gene_A': [100, 120, 95],
    'gene_B': [85, 90, 110],
    'gene_C': [150, 140, 160]
})

print("Wide (gene expression matrix):")
df_multi_wide

In [ ]:
# Melt to long
df_multi_long = pd.melt(
    df_multi_wide,
    id_vars=['subject'],
    value_vars=['gene_A', 'gene_B', 'gene_C'],
    var_name='gene',
    value_name='expression'
)

print("Long format (tidy):")
df_multi_long

## 5. Concatenating DataFrames

### Vertical concatenation (stacking rows)

In [ ]:
# Create separate session data
session1 = pd.DataFrame({
    'subject': ['S01', 'S02', 'S03'],
    'rt': [520, 498, 545],
    'accuracy': [0.85, 0.90, 0.88]
})

session2 = pd.DataFrame({
    'subject': ['S01', 'S02', 'S03'],
    'rt': [495, 478, 520],
    'accuracy': [0.92, 0.95, 0.90]
})

# Vertical concatenation
df_combined = pd.concat([session1, session2], ignore_index=True)

print("Combined sessions:")
df_combined

### Add source identifier

In [ ]:
# Add session identifier before concatenating
session1['session'] = 1
session2['session'] = 2

df_with_session = pd.concat([session1, session2], ignore_index=True)

print("With session identifier:")
df_with_session

### Horizontal concatenation (side-by-side)

In [ ]:
# Create behavioral and neural data
behavioral = pd.DataFrame({
    'trial': [1, 2, 3, 4, 5],
    'rt': [520, 498, 545, 501, 523],
    'accuracy': [1, 1, 0, 1, 1]
})

neural = pd.DataFrame({
    'firing_rate': [45, 52, 38, 48, 50],
    'spike_count': [22, 26, 19, 24, 25]
})

# Horizontal concatenation
df_combined_horiz = pd.concat([behavioral, neural], axis=1)

print("Horizontal concatenation:")
df_combined_horiz

## 6. Merging and Joining

In [ ]:
# Create example datasets
df_trials = pd.DataFrame({
    'trial_id': [1, 2, 3, 4, 5],
    'subject_id': ['S01', 'S01', 'S02', 'S02', 'S03'],
    'rt': [520, 498, 545, 501, 523],
    'accuracy': [1, 1, 0, 1, 1]
})

df_subjects = pd.DataFrame({
    'subject_id': ['S01', 'S02', 'S03', 'S04'],
    'age': [25, 30, 28, 32],
    'group': ['control', 'drug', 'control', 'drug']
})

print("Trial data:")
print(df_trials)
print("\nSubject data:")
print(df_subjects)

### Inner join (only matching rows)

In [ ]:
# Inner join
merged_inner = pd.merge(df_trials, df_subjects, on='subject_id', how='inner')

print("Inner join (only subjects present in both):")
merged_inner

### Left join (keep all left)

In [ ]:
# Left join
merged_left = pd.merge(df_trials, df_subjects, on='subject_id', how='left')

print("Left join (all trials kept):")
merged_left

### Right join (keep all right)

In [ ]:
# Right join
merged_right = pd.merge(df_trials, df_subjects, on='subject_id', how='right')

print("Right join (all subjects kept):")
merged_right

### Outer join (keep all from both)

In [ ]:
# Outer join
merged_outer = pd.merge(df_trials, df_subjects, on='subject_id', how='outer')

print("Outer join (everything kept):")
merged_outer

### Merging on different column names

In [ ]:
# Different column names
df1 = pd.DataFrame({
    'subject': ['S01', 'S02', 'S03'],
    'rt': [520, 545, 523]
})

df2 = pd.DataFrame({
    'participant_id': ['S01', 'S02', 'S03'],
    'age': [25, 30, 28]
})

# Merge with different column names
merged = pd.merge(df1, df2, left_on='subject', right_on='participant_id')

print("Merge with different column names:")
merged

## Summary

In this notebook we covered:
- GroupBy operations (single/multiple variables, iteration)
- Advanced aggregation (multiple aggregations, custom functions, transform, filter)
- Pivot tables and crosstabs
- Long/wide format conversion (melt, pivot)
- Concatenating DataFrames (vertical/horizontal)
- Merging DataFrames (inner/left/right/outer joins)

**Next notebook:** Signal Processing and Timeseries

In [ ]:
print("Notebook 3 complete!")